# System B (Single-Verifier Pipeline) — Standalone Evaluation

Runs System B (`AMG_RAG_System`) on its own against a labeled test set of clinical reports and reports Accuracy, Precision, Recall, F1, AUROC, confusion matrix, and latency. This is the same System B implementation and test set used in the side-by-side comparison notebook, extracted so it can be run and evaluated independently.


### 1. Install dependencies

In [1]:
%pip install -q "google-auth==2.49.0" "langchain-google-genai" "langchain-classic" "langchain-huggingface" "langchain-chroma" networkx wikipedia python-decouple
%pip install -q pandas tqdm scikit-learn matplotlib seaborn numpy

### 2. API key

Uses the same `GOOGLE_API_KEY` / Gemini-Gemma backend as System A.

In [2]:
import os
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your GOOGLE_API_KEY: ")

pubmed_key = getpass("Enter your PubMed API key (optional, press Enter to skip): ")
if pubmed_key:
    os.environ["pubmed_api"] = pubmed_key

GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
print("API key configured.")


Enter your GOOGLE_API_KEY: ··········
Enter your PubMed API key (optional, press Enter to skip): ··········
API key configured.


### 3. System B source

In [3]:
"""
AMG-RAG: Autonomous Medical Knowledge Graph RAG System
Complete implementation with dynamic KG generation and medical QA
"""

import json
import os
import time
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
import networkx as nx
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import requests
from xml.etree import ElementTree as ET
import wikipedia
from typing_extensions import TypedDict
from decouple import config
# Configuration - Replace with your API keys
GOOGLE_API_KEY = config('GOOGLE_API_KEY', default=os.environ.get('GOOGLE_API_KEY'))
PUBMED_API_KEY = config('pubmed_api', default=os.environ.get('pubmed_api') or None)

@dataclass
class MedicalEntity:
    """Represents a medical entity in the knowledge graph"""
    name: str
    description: str
    entity_type: str  # drug, disease, symptom, treatment, etc.
    confidence: float = 1.0
    sources: List[str] = field(default_factory=list)

@dataclass
class MedicalRelation:
    """Represents a relationship between medical entities"""
    source: str
    target: str
    relation_type: str
    confidence: float
    evidence: str
    sources: List[str] = field(default_factory=list)

class MedicalKnowledgeGraph:
    """Dynamic Medical Knowledge Graph with confidence scoring"""

    def __init__(self):
        self.graph = nx.DiGraph()
        self.entities = {}
        self.relations = []

    def add_entity(self, entity: MedicalEntity):
        """Add a medical entity to the graph"""
        self.entities[entity.name] = entity
        self.graph.add_node(
            entity.name,
            description=entity.description,
            entity_type=entity.entity_type,
            confidence=entity.confidence,
            sources=entity.sources
        )

    def add_relation(self, relation: MedicalRelation):
        """Add a relationship between entities"""
        self.relations.append(relation)
        self.graph.add_edge(
            relation.source,
            relation.target,
            relation_type=relation.relation_type,
            confidence=relation.confidence,
            evidence=relation.evidence,
            sources=relation.sources
        )

    def get_connected_nodes(self, node_name: str, confidence_threshold: float = 0.5):
        """Get nodes connected to a given node with confidence above threshold"""
        connected = []
        if node_name in self.graph:
            for neighbor in self.graph.neighbors(node_name):
                edge_data = self.graph[node_name][neighbor]
                if edge_data.get('confidence', 0) >= confidence_threshold:
                    connected.append({
                        'node': neighbor,
                        'relation': edge_data.get('relation_type'),
                        'confidence': edge_data.get('confidence'),
                        'evidence': edge_data.get('evidence')
                    })
        return connected

    def explore_path(self, start_node: str, max_depth: int = 3,
                    confidence_threshold: float = 0.5):
        """Explore paths from a starting node with confidence propagation"""
        paths = []
        visited = set()

        def dfs(node, path, accumulated_confidence, depth):
            if depth > max_depth or node in visited:
                return

            visited.add(node)

            if len(path) > 0:
                paths.append({
                    'path': path.copy(),
                    'confidence': accumulated_confidence,
                    'final_node': node
                })

            for neighbor_data in self.get_connected_nodes(node, confidence_threshold):
                neighbor = neighbor_data['node']
                new_confidence = accumulated_confidence * neighbor_data['confidence']

                if new_confidence >= confidence_threshold:
                    new_path = path + [(node, neighbor, neighbor_data['relation'])]
                    dfs(neighbor, new_path, new_confidence, depth + 1)

            visited.remove(node)

        dfs(start_node, [], 1.0, 0)
        return paths

class PubMedSearcher:
    """PubMed API wrapper for medical literature search"""

    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key
        self.base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

    def search(self, query: str, max_results: int = 3) -> List[str]:
        """Search PubMed and return article abstracts"""
        # Respect rate limits for free tier (3 requests/sec max)
        time.sleep(0.5)

        # Search for PMIDs
        search_url = f"{self.base_url}/esearch.fcgi"
        search_params = {
            "db": "pubmed",
            "term": query,
            "retmode": "xml",
            "retmax": max_results
        }
        if self.api_key:
            search_params["api_key"] = self.api_key

        try:
            response = requests.get(search_url, params=search_params, timeout=30)
            if response.status_code != 200:
                print(f"PubMed API search returned HTTP status {response.status_code}")
                return []

            if not response.text.strip().startswith("<?xml") and not response.text.strip().startswith("<"):
                print(f"PubMed API returned non-XML response during search: {response.text[:200]}")
                return []

            root = ET.fromstring(response.text)
            pmids = [id_elem.text for id_elem in root.findall(".//Id")]

            if not pmids:
                return []

            # Fetch abstracts
            time.sleep(0.5)
            fetch_url = f"{self.base_url}/efetch.fcgi"
            fetch_params = {
                "db": "pubmed",
                "id": ",".join(pmids),
                "retmode": "text",
                "rettype": "abstract"
            }
            if self.api_key:
                fetch_params["api_key"] = self.api_key

            response = requests.get(fetch_url, params=fetch_params, timeout=30)
            if response.status_code != 200:
                print(f"PubMed API fetch returned HTTP status {response.status_code}")
                return []

            articles = response.text.split("\n\n")

            # Clean and return abstracts
            abstracts = []
            for article in articles:
                lines = article.split("\n")
                abstract_lines = [line for line in lines if line.strip()
                                and not any(skip in line.lower() for skip in
                                          ["author", "doi", "pmid", "copyright"])]
                if abstract_lines:
                    abstracts.append(" ".join(abstract_lines))

            return abstracts

        except Exception as e:
            print(f"PubMed search error: {e}")
            return []

class AMG_RAG_System:
    """Main AMG-RAG system for medical question answering"""

    def __init__(self, google_api_key: str = None):
        # Initialize LLM with the same Gemini/Gemma backend and model as
        # System A, so both systems are evaluated on identical LLM output.
        if google_api_key:
            self.llm = ChatGoogleGenerativeAI(
                model="gemma-4-31b-it",
                temperature=0.0,
                google_api_key=google_api_key,
            )
        else:
            raise ValueError("Google API key is required.")

        # Initialize components
        self.kg = MedicalKnowledgeGraph()
        self.pubmed = PubMedSearcher(api_key=PUBMED_API_KEY)
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.vector_store = Chroma(
            collection_name="medical_qa",
            embedding_function=self.embeddings
        )

        # Initialize chains
        self._setup_chains()

    def _setup_chains(self):
        """Setup LLM chains for various tasks"""

        # Enhanced medical entity extraction with relevance scoring
        entity_schemas = [
            ResponseSchema(
                name="entities",
                description="List of medical entities (diseases, drugs, symptoms, treatments)",
                type="array"
            ),
            ResponseSchema(
                name="scores",
                description="Relevance scores (1-10) for each entity based on importance to the question",
                type="array"
            ),
            ResponseSchema(
                name="descriptions",
                description="Brief descriptions of each entity in the context of the question",
                type="array"
            )
        ]
        entity_parser = StructuredOutputParser.from_response_schemas(entity_schemas)

        self.entity_extractor = PromptTemplate(
            template="""Extract all medical entities from this question and options with relevance scoring.
            Include diseases, drugs, symptoms, treatments, and medical concepts.

            Question: {question}
            Options: {options}
            Context: {context}

            For each entity, provide:
            1. Entity name
            2. Relevance score (1-10): 10=directly related to question, 7-9=moderately relevant, 4-6=weakly relevant, 1-3=minimally relevant
            3. Brief description in context of the question

            Return in JSON format:
            {format_instructions}""",
            input_variables=["question", "options", "context"],
            partial_variables={"format_instructions": entity_parser.get_format_instructions()}
        ) | self.llm | entity_parser

        # Enhanced relation extraction with bidirectional analysis
        relation_schemas = [
            ResponseSchema(
                name="relationships",
                description="List of relationship dictionaries between entities containing entityA, entityB, relationship_type, confidence_score, and evidence",
                type="array"
            )
        ]
        relation_parser = StructuredOutputParser.from_response_schemas(relation_schemas)

        self.relation_extractor = PromptTemplate(
            template="""Analyze the medical relationships between these entities based on the context.

            Entities and Descriptions:
            {entities_with_descriptions}

            Context: {context}

            Identify and return all valid medical relationships between any of the given entities based on the medical context.

            Provide relationships in this exact JSON format:
            {{
                "relationships": [
                    {{
                        "entityA": "Entity Name 1",
                        "entityB": "Entity Name 2",
                        "relationship_type": "relationship_type_here",
                        "confidence_score": 8,
                        "evidence": "brief evidence here"
                    }}
                ]
            }}

            Use medical relationship types like: treats, causes, symptom_of, risk_factor_for, contraindicated_with, differential_diagnosis, etc.
            Confidence scores: 10=strong evidence, 7-9=moderate evidence, 4-6=weak evidence, 1-3=minimal evidence

            Return ONLY the JSON, no other text:""",
            input_variables=["entities_with_descriptions", "context"],
            partial_variables={"format_instructions": relation_parser.get_format_instructions()}
        ) | self.llm | relation_parser

        # Entity summarization chain
        summary_schemas = [
            ResponseSchema(
                name="summaries",
                description="Concise summaries for each entity based on context",
                type="array"
            ),
            ResponseSchema(
                name="scores",
                description="Relevance scores (1-10) for each summary",
                type="array"
            )
        ]
        summary_parser = StructuredOutputParser.from_response_schemas(summary_schemas)

        self.summary_chain = PromptTemplate(
            template="""Generate concise and relevant summaries for each medical entity based on the given context.

            Entities: {entities}
            Context: {context}

            For each entity, provide:
            1. A concise summary (2-3 sentences) focusing on relevance to the medical question
            2. Relevance score (1-10): 10=directly relevant, 7-9=moderately relevant, 4-6=weakly relevant, 1-3=minimally relevant

            Return in JSON format:
            {format_instructions}""",
            input_variables=["entities", "context"],
            partial_variables={"format_instructions": summary_parser.get_format_instructions()}
        ) | self.llm | summary_parser

        # Chain of thought reasoning
        cot_schemas = [
            ResponseSchema(
                name="reasoning",
                description="Step-by-step medical reasoning",
                type="string"
            )
        ]
        cot_parser = StructuredOutputParser.from_response_schemas(cot_schemas)

        self.cot_chain = PromptTemplate(
            template="""Based on the medical knowledge graph information and search results,
            provide step-by-step reasoning for this medical question.

            Question: {question}

            Graph Knowledge:
            {graph_context}

            Search Results:
            {search_context}

            Provide detailed medical reasoning:
            {format_instructions}""",
            input_variables=["question", "graph_context", "search_context"],
            partial_variables={"format_instructions": cot_parser.get_format_instructions()}
        ) | self.llm | cot_parser

        # Final answer generation
        answer_schemas = [
            ResponseSchema(
                name="answer",
                description="Final answer (A, B, C, D, or E)",
                type="string"
            ),
            ResponseSchema(
                name="confidence",
                description="Confidence in the answer (0-1)",
                type="number"
            ),
            ResponseSchema(
                name="explanation",
                description="Brief explanation",
                type="string"
            )
        ]
        answer_parser = StructuredOutputParser.from_response_schemas(answer_schemas)

        self.answer_chain = PromptTemplate(
            template="""Based on the reasoning and evidence, select the best answer.

            Question: {question}
            Options: {options}

            Reasoning:
            {reasoning}

            Evidence:
            {evidence}

            Select the best answer (A, B, C, D, or E):
            {format_instructions}""",
            input_variables=["question", "options", "reasoning", "evidence"],
            partial_variables={"format_instructions": answer_parser.get_format_instructions()}
        ) | self.llm | answer_parser

        # Report findings extractor
        finding_schemas = [
            ResponseSchema(
                name="findings",
                description="List of clinical claims/findings parsed from the report (e.g. 'heart is of normal size', 'lungs are clear')",
                type="array"
            ),
            ResponseSchema(
                name="anatomical_targets",
                description="The anatomical structures associated with each finding (e.g. 'heart', 'lungs', 'mediastinum')",
                type="array"
            ),
            ResponseSchema(
                name="clinical_status",
                description="Clinical observation status for each finding (e.g. 'normal', 'abnormal', 'clear')",
                type="array"
            )
        ]
        finding_parser = StructuredOutputParser.from_response_schemas(finding_schemas)

        self.finding_extractor = PromptTemplate(
            template="""Extract all individual medical findings/claims and associated anatomical structures from this clinical report.

            Report: {report}

            For each finding/claim, provide:
            1. The finding statement
            2. The anatomical target structure
            3. The clinical status (e.g. normal, abnormal, clear, consolidated, etc.)

            Return in JSON format:
            {format_instructions}""",
            input_variables=["report"],
            partial_variables={"format_instructions": finding_parser.get_format_instructions()}
        ) | self.llm | finding_parser

        # Report verifier schema
        verifier_schemas = [
            ResponseSchema(
                name="grounding_scores",
                description="Grounding/confidence score (0.0 to 1.0) for each finding based on clinical consistency and supporting context. 1.0 = highly plausible/normal medical finding, 0.5 = questionable/vague, 0.0 = highly contradictory or medically impossible.",
                type="array"
            ),
            ResponseSchema(
                name="assessments",
                description="Brief clinical validation reasoning for each finding's score, checking if the claim aligns with typical medical findings and standard terminology.",
                type="array"
            ),
            ResponseSchema(
                name="hallucination_indicators",
                description="Boolean flag for each finding: true if the finding appears contradictory, clinically impossible, or likely an LLM hallucination; false otherwise.",
                type="array"
            )
        ]
        verifier_parser = StructuredOutputParser.from_response_schemas(verifier_schemas)

        self.finding_verifier = PromptTemplate(
            template="""Validate the following clinical findings/claims against the provided literature context and general medical knowledge.
            Determine if they are medically consistent, physiologically plausible, and well-grounded.

            Findings: {findings_list}
            Literature & Evidence Context: {context}

            For each finding, provide:
            1. Grounding score (0.0 to 1.0): 1.0 = standard, highly consistent finding; 0.7-0.9 = plausible; 0.4-0.6 = weak grounding or inconsistent; 0.0-0.3 = clinically contradictory or medical hallucination.
            2. Brief assessment reasoning.
            3. Hallucination indicator (true/false).

            Return in JSON format:
            {format_instructions}""",
            input_variables=["findings_list", "context"],
            partial_variables={"format_instructions": verifier_parser.get_format_instructions()}
        ) | self.llm | verifier_parser

    def build_knowledge_graph(self, question: str, options: Dict[str, str]) -> None:
        """Build a dynamic knowledge graph for the question with enhanced entity extraction"""

        # Prepare context for entity extraction
        options_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
        full_text = question + " " + " ".join(options.values())

        # Search for additional context
        search_query = question + " " + " ".join(list(options.values())[:3])
        search_results = self.pubmed.search(search_query, max_results=3)
        context = "\n".join(search_results) if search_results else ""

        # Extract medical entities with relevance scoring
        try:
            entities_result = self.entity_extractor.invoke({
                "question": question,
                "options": options_text,
                "context": context
            })
            entities = entities_result.get("entities", [])
            scores = entities_result.get("scores", [])
            descriptions = entities_result.get("descriptions", [])
        except Exception as e:
            print(f"Entity extraction error: {e}")
            entities = list(options.values())[:3]  # Fallback to options
            scores = [5] * len(entities)  # Default moderate relevance
            descriptions = [f"Medical concept: {entity}" for entity in entities]

        print(f"Extracted entities: {entities}")
        print(f"Relevance scores: {scores}")

        # Add entities to graph with relevance-based confidence
        for i, entity in enumerate(entities[:8]):  # Limit to 8 entities
            # Search PubMed for additional information
            abstracts = self.pubmed.search(entity, max_results=2)

            # Search Wikipedia as fallback
            wiki_content = ""
            try:
                wiki_results = wikipedia.search(entity, results=1)
                if wiki_results:
                    wiki_content = wikipedia.summary(wiki_results[0], sentences=3)
            except:
                pass

            # Combine sources with LLM-generated description
            llm_description = descriptions[i] if i < len(descriptions) else f"Medical entity: {entity}"
            external_description = " ".join(abstracts) if abstracts else wiki_content
            combined_description = f"{llm_description}. {external_description}" if external_description else llm_description

            # Calculate confidence based on relevance score and external sources
            relevance_score = scores[i] if i < len(scores) else 5
            confidence = min(1.0, (relevance_score / 10.0) + (0.2 if abstracts else 0.1))

            # Add entity to graph
            med_entity = MedicalEntity(
                name=entity,
                description=combined_description[:500],  # Limit description length
                entity_type="medical_concept",
                confidence=confidence,
                sources=["PubMed", "Wikipedia"] if abstracts else ["Wikipedia"]
            )
            self.kg.add_entity(med_entity)

        # Extract relationships between entities in one batch for maximum speed
        entity_list = list(self.kg.entities.keys())
        if len(entity_list) > 1:
            try:
                # Prepare descriptions block
                desc_list = []
                for entity in entity_list:
                    desc_list.append(f"- {entity}: {self.kg.entities[entity].description[:200]}")
                entities_with_descriptions = "\n".join(desc_list)

                relationship_context = f"Question: {question}\n\nOptions: {options_text}\n\nSearch Results: {context}"

                print("Extracting relationships in a single batch...")
                relation_result = self.relation_extractor.invoke({
                    "entities_with_descriptions": entities_with_descriptions,
                    "context": relationship_context
                })

                relationships = relation_result.get("relationships", [])

                # Process each relationship in the list
                for rel in relationships:
                    if isinstance(rel, dict):
                        rel_type = rel.get("relationship_type", "related_to")
                        confidence = rel.get("confidence_score", 5) / 10.0
                        evidence = rel.get("evidence", "")
                        entity_a = rel.get("entityA", "")
                        entity_b = rel.get("entityB", "")

                        # Validate that both entities exist in our graph to avoid hallucinations
                        if entity_a in self.kg.entities and entity_b in self.kg.entities:
                            relation = MedicalRelation(
                                source=entity_a,
                                target=entity_b,
                                relation_type=rel_type,
                                confidence=confidence,
                                evidence=evidence,
                                sources=["LLM Analysis"]
                            )
                            self.kg.add_relation(relation)

            except Exception as e:
                print(f"Relation extraction error: {e}")

        # Generate entity summaries for better context
        self._generate_entity_summaries(question, context)

    def _generate_entity_summaries(self, question: str, context: str) -> None:
        """Generate enhanced summaries for entities in the knowledge graph"""
        if not self.kg.entities:
            return

        try:
            entities_list = list(self.kg.entities.keys())
            summary_result = self.summary_chain.invoke({
                "entities": entities_list,
                "context": f"Question: {question}\n\nContext: {context}"
            })

            summaries = summary_result.get("summaries", [])
            scores = summary_result.get("scores", [])

            # Update entity descriptions with enhanced summaries
            for i, entity_name in enumerate(entities_list):
                if i < len(summaries) and i < len(scores):
                    # Combine original description with enhanced summary
                    original_desc = self.kg.entities[entity_name].description
                    enhanced_summary = summaries[i]
                    relevance_score = scores[i]

                    # Update description with enhanced summary
                    updated_description = f"{original_desc}\n\nEnhanced Summary: {enhanced_summary}"

                    # Update confidence based on summary relevance
                    current_confidence = self.kg.entities[entity_name].confidence
                    summary_confidence = min(1.0, relevance_score / 10.0)
                    updated_confidence = min(1.0, (current_confidence + summary_confidence) / 2)

                    # Update the entity
                    self.kg.entities[entity_name].description = updated_description[:500]
                    self.kg.entities[entity_name].confidence = updated_confidence

        except Exception as e:
            print(f"Entity summarization error: {e}")

    def reason_with_graph(self, question: str, options: Dict[str, str]) -> Dict[str, Any]:
        """Perform reasoning using the knowledge graph"""

        # Explore graph paths for each entity
        graph_context = []
        for entity in list(self.kg.entities.keys())[:3]:
            # Get connected nodes
            connections = self.kg.get_connected_nodes(entity, confidence_threshold=0.3)

            # Explore paths
            paths = self.kg.explore_path(entity, max_depth=2, confidence_threshold=0.3)

            context = f"Entity: {entity}\n"
            context += f"Description: {self.kg.entities[entity].description[:200]}\n"

            if connections:
                context += "Direct connections:\n"
                for conn in connections[:3]:
                    context += f"  - {conn['relation']} -> {conn['node']} (confidence: {conn['confidence']:.2f})\n"

            if paths:
                context += "Reasoning paths:\n"
                for path_data in paths[:2]:
                    path_str = " -> ".join([f"{p[0]} [{p[2]}]" for p in path_data['path']])
                    if path_str:
                        context += f"  - {path_str} -> {path_data['final_node']} (confidence: {path_data['confidence']:.2f})\n"

            graph_context.append(context)

        # Search for additional evidence
        search_query = question + " " + " ".join(list(self.kg.entities.keys())[:3])
        search_results = self.pubmed.search(search_query, max_results=2)
        search_context = "\n".join(search_results) if search_results else "No additional search results found."

        # Generate chain of thought reasoning
        try:
            cot_result = self.cot_chain.invoke({
                "question": question,
                "graph_context": "\n\n".join(graph_context),
                "search_context": search_context
            })
            reasoning = cot_result.get("reasoning", "Unable to generate reasoning")
        except Exception as e:
            print(f"CoT generation error: {e}")
            reasoning = "Error in reasoning generation"

        # Generate final answer
        options_str = "\n".join([f"{k}: {v}" for k, v in options.items()])
        evidence = "\n".join(graph_context[:2])

        try:
            answer_result = self.answer_chain.invoke({
                "question": question,
                "options": options_str,
                "reasoning": reasoning,
                "evidence": evidence
            })

            return {
                "answer": answer_result.get("answer", "Unable to determine"),
                "confidence": answer_result.get("confidence", 0.0),
                "explanation": answer_result.get("explanation", ""),
                "reasoning": reasoning,
                "graph_context": graph_context,
                "search_context": search_context
            }
        except Exception as e:
            print(f"Answer generation error: {e}")
            return {
                "answer": "Error",
                "confidence": 0.0,
                "explanation": str(e),
                "reasoning": reasoning,
                "graph_context": graph_context,
                "search_context": search_context
            }

    def evaluate_medical_report(self, report: str) -> Dict[str, Any]:
        """Evaluate an LLM-generated medical report for clinical consistency, hallucinations, and confidence scoring."""
        print(f"\n{'='*50}")
        print("Step 1: Parsing report into clinical claims...")

        # 1. Extract findings
        finding_extraction_error_msg = None
        try:
            finding_result = self.finding_extractor.invoke({"report": report})
            findings = finding_result.get("findings", [])
            anatomical_targets = finding_result.get("anatomical_targets", [])
            clinical_statuses = finding_result.get("clinical_status", [])
        except Exception as e:
            finding_extraction_error_msg = f"{type(e).__name__}: {e}"
            print(f"Finding extraction error ({finding_extraction_error_msg}) -- falling back to naive sentence split.")
            # Fallback parsing
            findings = [s.strip() for s in report.split(".") if s.strip()]
            anatomical_targets = ["General"] * len(findings)
            clinical_statuses = ["Unspecified"] * len(findings)

        # Open-weight/prompt-based JSON extraction doesn't guarantee the three
        # parallel arrays come back the same length (unlike a strict schema-
        # enforced provider). Truncate to the shortest list and pad any gaps
        # with safe defaults so every downstream index is always in range.
        n = min(len(findings), len(anatomical_targets), len(clinical_statuses)) if findings and anatomical_targets and clinical_statuses else 0
        if n < len(findings):
            print(f"Warning: extractor returned mismatched array lengths "
                  f"(findings={len(findings)}, targets={len(anatomical_targets)}, "
                  f"statuses={len(clinical_statuses)}) — truncating to {n} aligned claims.")
        findings = findings[:n]
        anatomical_targets = (anatomical_targets[:n] + ["General"] * n)[:n]
        clinical_statuses = (clinical_statuses[:n] + ["Unspecified"] * n)[:n]

        print(f"Extracted {len(findings)} clinical claims.")

        # 2. Build Report Knowledge Graph
        # Clear existing graph first to avoid contamination
        self.kg = MedicalKnowledgeGraph()

        for i, target in enumerate(anatomical_targets):
            if i < len(findings):
                # Add anatomical target as an entity with high confidence
                entity_name = target.title()
                if entity_name not in self.kg.entities:
                    med_entity = MedicalEntity(
                        name=entity_name,
                        description=f"Anatomical region: {entity_name}. Status in report: {clinical_statuses[i]}",
                        entity_type="anatomical_structure",
                        confidence=1.0,
                        sources=["Report Parser"]
                    )
                    self.kg.add_entity(med_entity)

                # Add the specific finding/claim
                finding_name = f"Claim_{i+1}"
                finding_entity = MedicalEntity(
                    name=finding_name,
                    description=findings[i],
                    entity_type="clinical_finding",
                    confidence=1.0,
                    sources=["Report Parser"]
                )
                self.kg.add_entity(finding_entity)

                # Link target to finding
                relation = MedicalRelation(
                    source=entity_name,
                    target=finding_name,
                    relation_type="has_finding",
                    confidence=1.0,
                    evidence=f"Status: {clinical_statuses[i]}",
                    sources=["Report Parser"]
                )
                self.kg.add_relation(relation)

        # 3. Retrieve Context & Validate each claim
        print("Step 2: Performing literature verification on claims...")

        # We search PubMed or Wiki for anatomical targets to get typical reference context
        search_queries = list(set(anatomical_targets))[:3]
        combined_search_query = " ".join(search_queries) + " normal chest x-ray"
        pubmed_articles = self.pubmed.search(combined_search_query, max_results=3)
        context = "\n".join(pubmed_articles) if pubmed_articles else "Standard anatomical reference context for normal chest findings."

        findings_formatted = "\n".join([f"- {findings[i]} (Target: {anatomical_targets[i]}, Status: {clinical_statuses[i]})" for i in range(len(findings))])

        def _safe_float(v, default=0.9):
            try:
                return float(v)
            except (TypeError, ValueError):
                return default

        def _safe_bool(v, default=False):
            if isinstance(v, bool):
                return v
            if isinstance(v, str):
                return v.strip().lower() in ("true", "yes", "1")
            return default

        verification_error_msg = None
        try:
            verifier_result = self.finding_verifier.invoke({
                "findings_list": findings_formatted,
                "context": context
            })
            grounding_scores = [_safe_float(s) for s in verifier_result.get("grounding_scores", [])]
            assessments = [str(a) for a in verifier_result.get("assessments", [])]
            hallucination_indicators = [_safe_bool(h) for h in verifier_result.get("hallucination_indicators", [])]
        except Exception as e:
            verification_error_msg = f"{type(e).__name__}: {e}"
            print(f"Finding verification error ({verification_error_msg}) -- falling back to default 'not hallucinated' for this report.")
            grounding_scores = [0.9] * len(findings)
            assessments = ["Standard verification fallback (LLM call failed)."] * len(findings)
            hallucination_indicators = [False] * len(findings)

        # 4. Compile detailed results and update KG confidence
        detailed_findings = []
        for i in range(len(findings)):
            g_score = grounding_scores[i] if i < len(grounding_scores) else 0.9
            assessment = assessments[i] if i < len(assessments) else "No detailed assessment."
            is_hallucination = hallucination_indicators[i] if i < len(hallucination_indicators) else False

            detailed_findings.append({
                "finding": findings[i],
                "target": anatomical_targets[i],
                "status": clinical_statuses[i],
                "grounding_score": g_score,
                "assessment": assessment,
                "hallucination_risk": is_hallucination
            })

            # Update finding entity confidence in KG
            finding_name = f"Claim_{i+1}"
            if finding_name in self.kg.entities:
                self.kg.entities[finding_name].confidence = g_score
                self.kg.entities[finding_name].description += f" (Verified Score: {g_score:.2f} - {assessment})"

            # Update connection edge confidence in KG
            target_name = anatomical_targets[i].title()
            if target_name in self.kg.graph and finding_name in self.kg.graph[target_name]:
                self.kg.graph[target_name][finding_name]['confidence'] = g_score

        # Calculate overall report confidence
        if detailed_findings:
            overall_confidence = sum([f["grounding_score"] for f in detailed_findings]) / len(detailed_findings)
            hallucination_detected = any([f["hallucination_risk"] for f in detailed_findings])
        else:
            overall_confidence = 0.0
            hallucination_detected = False

        return {
            "report": report,
            "overall_confidence": overall_confidence,
            "hallucination_detected": hallucination_detected,
            "detailed_findings": detailed_findings,
            "finding_extraction_error": finding_extraction_error_msg,
            "verification_error": verification_error_msg,
            "graph_stats": {
                "num_entities": len(self.kg.entities),
                "num_relations": len(self.kg.relations)
            }
        }

    def answer_question(self, question_data: Dict[str, Any]) -> Dict[str, Any]:
        """Main pipeline to answer a medical question"""

        question = question_data["question"]
        options = question_data.get("options", {})

        print(f"\n{'='*50}")
        print(f"Question: {question}")
        print(f"Options: {options}")
        print(f"{'='*50}\n")

        # Step 1: Build knowledge graph
        print("Step 1: Building knowledge graph...")
        self.build_knowledge_graph(question, options)
        print(f"Graph has {len(self.kg.entities)} entities and {len(self.kg.relations)} relations")

        # Step 2: Reason with graph
        print("\nStep 2: Reasoning with graph...")
        result = self.reason_with_graph(question, options)

        # Add metadata
        result["question"] = question
        result["options"] = options
        result["expected_answer"] = question_data.get("answer", "Unknown")
        result["graph_stats"] = {
            "num_entities": len(self.kg.entities),
            "num_relations": len(self.kg.relations)
        }

        return result

def load_medqa_sample():
    """Load a sample from MEDQA dataset"""
    # Sample MEDQA question
    sample = {
        "question": "A 45-year-old man presents to the emergency department with severe chest pain that started 2 hours ago. The pain is substernal, crushing in nature, and radiates to his left arm. He has a history of hypertension and diabetes mellitus. His father died of a myocardial infarction at age 50. On examination, he is diaphoretic and in distress. His blood pressure is 150/90 mmHg, pulse is 110/min, and respirations are 22/min. An ECG shows ST-segment elevation in leads II, III, and aVF. Which of the following is the most likely diagnosis?",
        "options": {
            "A": "Unstable angina",
            "B": "Acute inferior wall myocardial infarction",
            "C": "Acute anterior wall myocardial infarction",
            "D": "Aortic dissection",
            "E": "Pulmonary embolism"
        },
        "answer": "B",
        "answer_idx": 1,
        "meta_info": "This is a cardiology question testing knowledge of myocardial infarction presentation and ECG findings."
    }
    return sample

def main():
    """Main execution function"""

    print("AMG-RAG Medical QA System")
    print("="*50)

    # Initialize system
    print("Initializing AMG-RAG system...")

    # Using Google Gemini
    system = AMG_RAG_System(google_api_key=GOOGLE_API_KEY)

    # Load sample question
    print("\nLoading MEDQA sample question...")
    question_data = load_medqa_sample()



    result = system.answer_question(question_data)



    # Display results
    print("\n" + "="*50)
    print("RESULTS")
    print("="*50)
    print(f"Question: {result['question'][:100]}...")
    print(f"\nOptions:")
    for k, v in result['options'].items():
        print(f"  {k}: {v}")

    print(f"\nExpected Answer: {result['expected_answer']}")
    print(f"Model Answer: {result['answer']}")
    print(f"Confidence: {result['confidence']:.2f}")
    print(f"\nExplanation: {result['explanation']}")

    print(f"\nGraph Statistics:")
    print(f"  - Entities: {result['graph_stats']['num_entities']}")
    print(f"  - Relations: {result['graph_stats']['num_relations']}")

    print(f"\nReasoning Chain:")
    print(result['reasoning'][:500] + "..." if len(result['reasoning']) > 500 else result['reasoning'])



    # Visualize graph structure (text-based)
    print("\n" + "="*50)
    print("KNOWLEDGE GRAPH STRUCTURE")
    print("="*50)

    for entity_name, entity in list(system.kg.entities.items())[:5]:
        print(f"\n[*] {entity_name}")
        print(f"   Type: {entity.entity_type}")
        print(f"   Description: {entity.description[:100]}...")

        connections = system.kg.get_connected_nodes(entity_name)
        if connections:
            print("   Connections:")
            for conn in connections[:3]:
                print(f"     -> {conn['relation']} -> {conn['node']} (conf: {conn['confidence']:.2f})")

    # New section: Report verification demonstration
    print("\n" + "="*50)
    print("DEMONSTRATION: CLINICAL REPORT VERIFICATION & HALLUCINATION ASSESSMENT")
    print("="*50)

    sample_report = (
        "The heart is of normal size with no abnormalities. The lungs are clear with no signs of "
        "consolidation or fluid accumulation. The structures around the lungs, including the "
        "mediastinum and hilar regions are normal. Overall, there are no acute or concerning findings."
    )

    print(f"Input Chest X-ray Report:\n\"{sample_report}\"")

    report_result = system.evaluate_medical_report(sample_report)

    print("\n" + "="*50)
    print("REPORT VERIFICATION RESULTS")
    print("="*50)
    print(f"Overall Report Confidence: {report_result['overall_confidence']:.2f}")
    print(f"Hallucination / Critical Risk Detected: {report_result['hallucination_detected']}")

    print("\nDetailed Findings Breakdown:")
    for idx, item in enumerate(report_result['detailed_findings']):
        print(f"\nFinding #{idx+1}:")
        print(f"  - Statement: \"{item['finding']}\"")
        print(f"  - Anatomical Target: {item['target']}")
        print(f"  - Clinical Status: {item['status']}")
        print(f"  - Grounding/Confidence Score: {item['grounding_score']:.2f}")
        print(f"  - Verification Assessment: {item['assessment']}")
        print(f"  - Hallucination Risk: {item['hallucination_risk']}")

    print("\n" + "="*50)
    print("FINDINGS GRAPH STRUCTURE")
    print("="*50)
    for entity_name, entity in list(system.kg.entities.items()):
        print(f"\n[*] {entity_name} ({entity.entity_type})")
        print(f"   Confidence: {entity.confidence:.2f}")
        print(f"   Details: {entity.description[:150]}")
        connections = system.kg.get_connected_nodes(entity_name, confidence_threshold=0.0)
        if connections:
            print("   Connections:")
            for conn in connections:
                print(f"     -> {conn['relation']} -> {conn['node']} (conf: {conn['confidence']:.2f})")

    # Demonstration 2: Hallucinated/Contradictory Report
    print("\n" + "="*50)
    print("DEMONSTRATION 2: HALLUCINATED / CONTRADICTORY REPORT")
    print("="*50)

    hallucinated_report = (
        "The heart is of normal size with signs of severe acute cardiomegaly. The lungs show massive "
        "consolidations in the upper lobes with completely clear lung fields. The mediastinum is "
        "widened and perfectly normal."
    )

    print(f"Input Hallucinated Report:\n\"{hallucinated_report}\"")

    hallucinated_result = system.evaluate_medical_report(hallucinated_report)

    print("\n" + "="*50)
    print("REPORT VERIFICATION RESULTS (HALLUCINATED REPORT)")
    print("="*50)
    print(f"Overall Report Confidence: {hallucinated_result['overall_confidence']:.2f}")
    print(f"Hallucination / Critical Risk Detected: {hallucinated_result['hallucination_detected']}")

    print("\nDetailed Findings Breakdown:")
    for idx, item in enumerate(hallucinated_result['detailed_findings']):
        print(f"\nFinding #{idx+1}:")
        print(f"  - Statement: \"{item['finding']}\"")
        print(f"  - Anatomical Target: {item['target']}")
        print(f"  - Clinical Status: {item['status']}")
        print(f"  - Grounding/Confidence Score: {item['grounding_score']:.2f}")
        print(f"  - Verification Assessment: {item['assessment']}")
        print(f"  - Hallucination Risk: {item['hallucination_risk']}")

    print("\n" + "="*50)
    print("FINDINGS GRAPH STRUCTURE (HALLUCINATED)")
    print("="*50)
    for entity_name, entity in list(system.kg.entities.items()):
        print(f"\n[*] {entity_name} ({entity.entity_type})")
        print(f"   Confidence: {entity.confidence:.2f}")
        print(f"   Details: {entity.description[:150]}")
        connections = system.kg.get_connected_nodes(entity_name, confidence_threshold=0.0)
        if connections:
            print("   Connections:")
            for conn in connections:
                print(f"     -> {conn['relation']} -> {conn['node']} (conf: {conn['confidence']:.2f})")

# NOTE: main() is a self-contained CLI-style demo (MEDQA sample question +
# 2 hardcoded report demos). It's left here for reference but is NOT auto-run,
# so importing this cell doesn't trigger extra LLM calls / cost.
# Uncomment the next two lines if you want to run that demo:
# if __name__ == "__main__":
#     main()

# Initialize the system for use in the rest of this notebook.
# Uses the GOOGLE_API_KEY set in the cell above (edit that cell, not this one).
rag_system = AMG_RAG_System(google_api_key=GOOGLE_API_KEY)
print('AMG-RAG System successfully initialized!')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

AMG-RAG System successfully initialized!


In [4]:
system_b = AMG_RAG_System(google_api_key=GOOGLE_API_KEY)
print("System B initialized.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

System B initialized.


### 4. Labeled test set

In [5]:
test_reports = [
    # --- Category C: Impossible Anatomy / Surgical History (Label 1) ---
    {"report": "Patient is status post total abdominal hysterectomy and bilateral salpingo-oophorectomy. A 4 cm complex cystic lesion is seen within the right ovary.", "expected_label": 1},
    {"report": "History of bilateral total knee arthroplasty. Severe tricompartmental osteoarthritis and joint space narrowing are seen in the native right knee.", "expected_label": 1},
    {"report": "Status post right radical nephrectomy. Mild hydronephrosis is identified in the right kidney.", "expected_label": 1},
    {"report": "The patient has a known history of complete thyroidectomy. The thyroid gland is symmetrically enlarged and heterogenous.", "expected_label": 1},

    # --- Category D: Hard True Negatives (Label 0) ---
    {"report": "History of left lower lobectomy. Expected postsurgical changes are present with volume loss and elevation of the left hemidiaphragm. The remaining left lung parenchyma is clear.", "expected_label": 0},
    {"report": "There is no pulmonary edema. Prominent bibasilar reticular opacities are stable and chronic, consistent with the patient's known idiopathic pulmonary fibrosis.", "expected_label": 0},
    {"report": "Lungs are clear of acute consolidation. The previously seen right apical cavitary lesion has been surgically resected, with stable postoperative scarring and surgical clips now seen.", "expected_label": 0},
    {"report": "No acute intracranial hemorrhage or major vascular occlusion. Severe, confluent periventricular white matter hypoattenuation is noted, reflecting chronic small vessel ischemic disease.", "expected_label": 0},

    # --- Category E: Acuity & Temporal Mismatch (Label 1) ---
    {"report": "No acute cardiopulmonary process. There is a new, large right-sided pneumothorax with mild leftward mediastinal shift compared to the prior radiograph.", "expected_label": 1},
    {"report": "The chest is radiographically stable. Interval development of moderate pulmonary edema and bilateral pleural effusions is noted since the examination yesterday.", "expected_label": 1},
    {"report": "No acute findings are identified on today's study. There is a new, acute compression fracture of the L4 vertebral body with 50% height loss.", "expected_label": 1},

    # --- Category E: Acuity & Temporal Consistency (Label 0) ---
    {"report": "No acute cardiopulmonary process. Stable, chronic right-sided pleural thickening is noted, unchanged since the prior study from 2018.", "expected_label": 0},
    {"report": "No acute osseous abnormality. Old, well-healed wedge compression deformity of T12 is unchanged from multiple prior comparisons.", "expected_label": 0},

    # --- Category H: Negation-Scope & Hedging Traps (Label 1 & 0) ---
    {"report": "It cannot be definitively stated that there is no underlying mass, though none is visible on the current non-contrast study. Lungs are otherwise clear.", "expected_label": 0},
    {"report": "There is no radiologic evidence to exclude a tiny radiolucent foreign body, however, no opaque foreign body is definitively seen on this film.", "expected_label": 0},
    {"report": "I am entirely unable to see any evidence of an acute fracture on this study. A displaced, acute spiral fracture of the distal radius is clearly visualized.", "expected_label": 1},
    {"report": "We cannot rule out a subtle developing infiltrate in the clinical setting of fever; however, the lungs are completely opacified by massive bilateral effusions.", "expected_label": 1},

    # --- Category K: Congenital Variants & Anatomical Logic (Label 1 & 0) ---
    {"report": "An azygos lobe variant is present in the right upper lung zone, created by the lateral downward invagination of the azygos vein. Lungs are clear.", "expected_label": 0},
    {"report": "A horseshoe kidney is noted, crossing the midline inferiorly to fuse at the lower poles. Both kidneys are entirely separate and located in their normal independent retroperitoneal anatomical positions.", "expected_label": 1},
    {"report": "Uterus didelphys is incidentally noted, demonstrating two distinct uterine horns and two cervices, consistent with the known Mullerian duct anomaly.", "expected_label": 0},
    {"report": "The patient has a known right-sided aortic arch. The descending thoracic aorta is seen normally traversing the left side of the spine.", "expected_label": 1},

    # --- Mixed Edge Cases (Label 1 & 0) ---
    {"report": "The appendix is normal in caliber without evidence of acute appendicitis. A severely inflamed, dilated, and perforated appendix is surrounded by inflammatory stranding.", "expected_label": 1},
    {"report": "The endotracheal tube is in good position. The tip of the endotracheal tube is currently located in the right mainstem bronchus, resulting in complete collapse of the left lung.", "expected_label": 1}
]

print(f"Total complex test reports: {len(test_reports)}")
print(f"Positive (should flag): {sum(r['expected_label'] for r in test_reports)}")
print(f"Negative (should not flag): {sum(1 - r['expected_label'] for r in test_reports)}")

Total complex test reports: 23
Positive (should flag): 13
Negative (should not flag): 10


### 5. Run System B over the test set, timing each report

In [6]:
import time
import io
import contextlib
from tqdm import tqdm

system_b_results = []

for i, item in enumerate(tqdm(test_reports, desc="System B")):
    report_text = item["report"]
    expected = item["expected_label"]

    start = time.perf_counter()
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            result = system_b.evaluate_medical_report(report_text)
        elapsed = time.perf_counter() - start
        error = None
        predicted = int(result["hallucination_detected"])
        confidence = result.get("overall_confidence", None)
        # These are set internally whenever a sub-step's LLM call fails and the
        # pipeline silently falls back to a default value instead of raising --
        # that's why n_errors could read 0 even when the LLM is never
        # successfully called for verification. Surface them here instead of
        # letting them disappear along with the captured stdout.
        finding_extraction_error = result.get("finding_extraction_error")
        verification_error = result.get("verification_error")
    except Exception as e:
        elapsed = time.perf_counter() - start
        error = str(e)
        predicted = None
        confidence = None
        finding_extraction_error = None
        verification_error = None

    system_b_results.append({
        "report_id": i + 1,
        "report": report_text,
        "expected_label": expected,
        "predicted_label": predicted,
        "overall_confidence": confidence,
        "latency_sec": elapsed,
        "error": error,
        "finding_extraction_error": finding_extraction_error,
        "verification_error": verification_error,
        "captured_log": buf.getvalue(),
    })

n_hard_errors = sum(1 for r in system_b_results if r["error"])
n_verifier_fallback = sum(1 for r in system_b_results if r["verification_error"])
n_extractor_fallback = sum(1 for r in system_b_results if r["finding_extraction_error"])

print(f"Done. {n_hard_errors} report(s) errored out at the top level.")
print(f"{n_verifier_fallback}/{len(system_b_results)} report(s) had the finding-verifier LLM call fail internally "
      f"and silently fall back to 'not hallucinated' (this was invisible in n_errors before this patch).")
print(f"{n_extractor_fallback}/{len(system_b_results)} report(s) had the finding-extractor LLM call fail internally.")

if n_verifier_fallback:
    print("\nSample verifier failure reasons (first 3 distinct):")
    seen = set()
    for r in system_b_results:
        if r["verification_error"] and r["verification_error"] not in seen:
            seen.add(r["verification_error"])
            print(f"  - {r['verification_error']}")
        if len(seen) >= 3:
            break


System B: 100%|██████████| 23/23 [22:03<00:00, 57.53s/it]

Done. 0 report(s) errored out at the top level.
0/23 report(s) had the finding-verifier LLM call fail internally and silently fall back to 'not hallucinated' (this was invisible in n_errors before this patch).
0/23 report(s) had the finding-extractor LLM call fail internally.


### 6. Save results to disk

In [7]:
import json

with open("system_b_results.json", "w") as f:
    json.dump(system_b_results, f, indent=2)

print("Saved system_b_results.json")


Saved system_b_results.json


### 7. Metrics — Accuracy, Precision, Recall, F1, AUROC, latency


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)

df_b = pd.DataFrame(system_b_results)
valid_b = df_b[df_b["error"].isna()].copy()

if len(valid_b) < len(df_b):
    print(f"WARNING: {len(df_b) - len(valid_b)} report(s) errored and are excluded:")
    print(df_b[df_b["error"].notna()][["report_id", "error"]])

y_true = valid_b["expected_label"].astype(int)
y_pred = valid_b["predicted_label"].astype(int)

metrics_b = {
    "system": "System B (Single-Verifier Pipeline)",
    "n_evaluated": len(valid_b),
    "n_errors": len(df_b) - len(valid_b),
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "mean_latency_sec": valid_b["latency_sec"].mean(),
    "median_latency_sec": valid_b["latency_sec"].median(),
    "p95_latency_sec": np.percentile(valid_b["latency_sec"], 95),
    "total_latency_sec": valid_b["latency_sec"].sum(),
}

# AUROC from the continuous confidence score (inverted: higher confidence score =
# lower predicted hallucination likelihood, so we flip it to score hallucination risk)
try:
    hallucination_risk_score = 1.0 - valid_b["overall_confidence"].astype(float)
    metrics_b["auroc"] = roc_auc_score(y_true, hallucination_risk_score)
except Exception as e:
    metrics_b["auroc"] = None
    print(f"AUROC could not be computed: {e}")

metrics_b["confusion_matrix"] = confusion_matrix(y_true, y_pred).tolist()

for k, v in metrics_b.items():
    if k != "confusion_matrix":
        print(f"{k}: {v}")
print("confusion_matrix (rows=actual, cols=predicted):")
print(np.array(metrics_b["confusion_matrix"]))


### 8. Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5, 4))
sns.heatmap(metrics_b["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not hallucinated", "Hallucinated"],
            yticklabels=["Not hallucinated", "Hallucinated"])
plt.title("System B — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


### 9. Classification metrics bar chart

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1", "auroc"]
metric_vals = [metrics_b[c] for c in metric_cols]

plt.figure(figsize=(7, 4))
sns.barplot(x=metric_cols, y=metric_vals, color="steelblue")
plt.title("System B — Classification Metrics")
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()


### 10. Latency breakdown

In [ ]:
latency_cols = ["mean_latency_sec", "median_latency_sec", "p95_latency_sec"]
latency_vals = [metrics_b[c] for c in latency_cols]

plt.figure(figsize=(6, 4))
sns.barplot(x=latency_cols, y=latency_vals, color="darkorange")
plt.title("System B — Latency (seconds)")
plt.tight_layout()
plt.show()

print(f"Total time to evaluate all {len(valid_b)} reports: {metrics_b['total_latency_sec']:.1f} s")


### 11. Per-report results and misclassifications

In [ ]:
display_cols = ["report_id", "report", "expected_label", "predicted_label", "overall_confidence", "latency_sec"]
valid_b[display_cols]


In [ ]:
misclassified = valid_b[valid_b["expected_label"] != valid_b["predicted_label"]]
print(f"Misclassified: {len(misclassified)} / {len(valid_b)}")
misclassified[display_cols]
